### Setup

In [1]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from common.utils import DataPreprocessor, FeatureEngineer, set_seed
from common.exp_data_utils import ExperimentDataPreprocessor
from common.eval import Evaluator

MOVIELENS_DATA_DIR = "../datasets/hetrec2011-movielens-2k-v2/user_ratedmovies.dat"
RANDOM_SEED = 42

# Initialize data processors
set_seed(RANDOM_SEED)
data_preprocessor = DataPreprocessor()
feature_engineer = FeatureEngineer()
experiment_data_preprocessor = ExperimentDataPreprocessor()
evaluator = Evaluator()


/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Seed set to 42
Seed set to 42


random seed set to 42
numpy seed set to 42
torch seed set to 42
lightning seed set to 42
torch set to use deterministic algorithms


### Load and Process DataFrame

In [2]:
interaction_df = data_preprocessor.load_and_process_df(
    file_dir=MOVIELENS_DATA_DIR,
    year_range=(2006, 2008),
)
interaction_df.head()

Data count: 855598
Data count after filtering by year (2006, 2008): 480608
Num of distinct users: 2103
Num of distinct items: 9519
done!
------------------------------
Filtering by min user/item interactions (10/0):
Data count before: 480608
Data count after: 480448
done!
------------------------------
==== Final Data Info: ====
Data Year Range: (2006, 2008)
Rating Threshold: 4.0
Num of interactions: 480448
Num of distinct users: 2064
Num of distinct items: 9519


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1


### Join Side Information

In [3]:
interaction_info_df = data_preprocessor.join_item_features(
    df=interaction_df, actor_k=5,
)
interaction_info_df.head()

extracting item features...
merging features...
interaction data count before merging: 480448
interaction data count after merging: 478404
done!


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label,actorID,country,directorID,directorName,genre
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0,"[jack_lemmon, walter_matthau, annmargret, burg...",USA,donald_petrie,Donald Petrie,"[Comedy, Romance, [PAD], [PAD], [PAD], [PAD], ..."
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1,"[bhiravi_vaidhy, dilip_satgare, haresh_mehta, ...",USA,siddharth_randeria,Siddharth Randeria,"[Sci-Fi, Thriller, [PAD], [PAD], [PAD], [PAD],..."
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1,"[mel_gibson, sophie_marceau, patrick_mcgoohan,...",USA,mel_gibson,Mel Gibson,"[Action, Drama, War, [PAD], [PAD], [PAD], [PAD..."
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0,"[dylan_walsh, laura_linney, ernie_hudson_jr, t...",USA,frank_marshall,Frank Marshall,"[Action, Adventure, Mystery, Sci-Fi, [PAD], [P..."
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1,"[antonio_banderas, salma_hayek, 1142520-joaqui...",USA,robert_rodriguez,Robert Rodriguez,"[Action, Romance, Thriller, [PAD], [PAD], [PAD..."


### Prepare Train/Valid/Test Set

In [4]:
# TODO: determine which method to use for splitting
# 1. Split by year
# 2. Stratified split by user, timestamp

train_df, valid_df, test_df = experiment_data_preprocessor.stratified_time_split(
    interaction_info_df,
    time_col="timestamp",
    train_ratio=0.75,
    val_ratio=0.1,
    test_ratio=0.15,
)

TRAIN_NUM_USERS = len(train_df["userID"].unique())
TRAIN_NUM_ITEMS = len(train_df["movieID"].unique())


Splitting data into train/valid/test by time period with ratio=(0.75 : 0.1 : 0.15):
train: 358027 (74.84%)
valid: 46916 (9.81%)
test: 73461 (15.36%)
------------------------------ 

Check target label distribution after splitting (%):
train label
0    0.555944
1    0.444056
Name: proportion, dtype: float64
valid label
0    0.610772
1    0.389228
Name: proportion, dtype: float64
test label
0    0.585277
1    0.414723
Name: proportion, dtype: float64


### Re-index User/Item ID & Encode Categorical Features

In [7]:
print("Train: fit_transform")
encoded_train_df = feature_engineer.fit_transform(train_df)
print("---"*10)
print("Valid: transform")
encoded_valid_df = feature_engineer.transform(valid_df)
print("---"*10)
print("Test: transform")
encoded_test_df = feature_engineer.transform(test_df)
print("---"*10)

Train: fit_transform
Re-index mapping dumped into ...
user: ../datasets/userid_mapping.csv
item: ../datasets/itemid_mapping.csv
Fitted: user/item mapping
Fitted: vocab2idx for actorID
Fitted: vocab2idx for country
Fitted: vocab2idx for directorID
Fitted: vocab2idx for genre
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Valid: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Test: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------


In [8]:
# # NOTE: can check the encoding vocab idx content from the feature engineer
# oov_idx = feature_engineer.vocab2idx["movieID"]["[OOV]"]
# len(test_df[test_df["movieID"] == oov_idx])

### Prepare Additional Data for Train/Inference

#### Build bi-partite graph for training

In [9]:
# NOTE: At training, we use interaction graph from train_df for train and validation
train_graph = experiment_data_preprocessor.create_interaction_graph(encoded_train_df)

# NOTE: At inference, we can use graph of (train_df + valid_df)
# train_valid_graph = utils.create_interaction_graph(pd.concat([train_df, valid_df], axis=0))

Creating interaction graph...
Drop negative samples
  Num of all interactions: 358027
  Num of positive interactions: 158984 

Building edges...
Building labels...
Interaction Graph: Data(edge_index=[2, 158984], edge_label=[158984])
Edge Index: tensor([[   0,    0,    0,  ..., 2063, 2063, 2063],
        [1102, 1186,  670,  ..., 2835,  742, 2969]])


#### Prepare train/valid triplet data

In [10]:
train_triplet_df = experiment_data_preprocessor.prepare_triplet_df(encoded_train_df, k_negative_samples=5)
valid_triplet_df = experiment_data_preprocessor.prepare_triplet_df(encoded_valid_df, k_negative_samples=5)
valid_triplet_df.head(1)

Original data count (positive samples): 158984
Num of triplets: 158984(pos samples) * 5(negative sampled items) = 794920
Original data count (positive samples): 18261
Num of triplets: 18261(pos samples) * 5(negative sampled items) = 91305


,userID,pos_item_id,neg_item_id,actorID_idx,country_idx,directorID_idx,genre_idx,neg_actorID_idx,neg_country_idx,neg_directorID_idx,neg_genre_idx
0,0,962,4662,"[3226, 5590, 10946, 7334, 13680]",63,2993,"[1, 2, 5, 9, 11, 0, 0, 0]","[7669, 5250, 15343, 5561, 7229]",63,583,"[8, 15, 0, 0, 0, 0, 0, 0]"


#### Prepare prediction pool for inference/testing

In [11]:
# NOTE: Prepare prediction pool to evaluate the model
prediction_pool_df = experiment_data_preprocessor.prepare_prediction_df(encoded_test_df, K=500)
prediction_pool_df.tail()

Prediction DataFrame:
User Pool: 2064
Item Pool: 6959, negative sampled to 500 items for each user
Num of interactions: 2064(users) * 500(items) = 1032000


,userID,movieID,label,actorID_idx,country_idx,directorID_idx,genre_idx
1031995,2063,447,0,"[15727, 2894, 10573, 253, 3728]",62,2985,"[8, 0, 0, 0, 0, 0, 0, 0]"
1031996,2063,3601,0,"[11716, 4113, 7140, 7336, 6954]",63,2854,"[1, 18, 0, 0, 0, 0, 0, 0]"
1031997,2063,7982,0,"[6627, 283, 3901, 11769, 14462]",63,1419,"[11, 0, 0, 0, 0, 0, 0, 0]"
1031998,2063,5969,0,"[11666, 10099, 4687, 6007, 6501]",63,3441,"[8, 15, 0, 0, 0, 0, 0, 0]"
1031999,2063,5025,0,"[3962, 7760, 5111, 6372, 4643]",62,558,"[6, 11, 14, 17, 0, 0, 0, 0]"


### Prepare DataLoader

In [12]:
# NOTE: ensure reproducibility of DataLoader
import torch
from common.utils import seed_worker
g = torch.Generator()
g.manual_seed(RANDOM_SEED)

# TODO: determine which Dataset to use
from torch.utils.data import DataLoader
from common.datasets import TripletDataset, UserItemPairDataset

BATCH_SIZE = 1024

train_dataset = TripletDataset(train_triplet_df)
valid_dataset = TripletDataset(valid_triplet_df)
test_dataset = UserItemPairDataset(prediction_pool_df)
print("train data count:", len(train_dataset))
print("valid data count:", len(valid_dataset))
print("test data count:", len(test_dataset))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, worker_init_fn=seed_worker, generator=g, num_workers=4)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


train data count: 794920
valid data count: 91305
test data count: 1032000


### Configure Model (LightningModule)

In [13]:
from lightning_models.gcn_cf_rec import GCNRec

EMB_DIM = 32
LR = 1e-3
EPOCHS = 50
NUM_LAYERS = 3

model = GCNRec(
    num_users=TRAIN_NUM_USERS,
    num_items=TRAIN_NUM_ITEMS,
    graph_data=train_graph,  # shape [2, num_edges]
    dim_id=EMB_DIM,
    num_layers=NUM_LAYERS,
    concat=True,
    lr=LR,
    reg_weight=1e-5,
)


### Configure Trainer and Experiment

In [14]:
from common._mlflow import get_mlflow_logger, get_callbacks

EXPERIMENT_NAME = "gcn-exp"
RUN_NAME = "run0"
PATIENCE = 5
mlflow_logger = get_mlflow_logger(experiment_name=EXPERIMENT_NAME, run_name=RUN_NAME)
trainer_callbacks = get_callbacks(
    exp_name=EXPERIMENT_NAME,
    run_name=RUN_NAME,
    patience=PATIENCE,
    monitor_metric="val_loss",
    monitor_mode="min",
    hyper_param_str=f"emb_dim={EMB_DIM}-num_layers={NUM_LAYERS}-lr={LR}-batch_size={BATCH_SIZE}-epochs={EPOCHS}",
)

In [15]:
from pytorch_lightning import Trainer

trainer = Trainer(
    max_epochs=EPOCHS,
    logger=mlflow_logger,
    log_every_n_steps=50,
    callbacks=trainer_callbacks,
    accelerator='cpu',  # or 'auto', 'gpu'
    # devices=[0], # if gpu is available
)


GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.


### Train Model

In [16]:
# Start training
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)



  | Name      | Type            | Params | Mode 
------------------------------------------------------
0 | gcn_model | GraphConvModule | 376 K  | train
1 | bpr_loss  | BPRLoss         | 0      | train
2 | reg_loss  | EmbLoss         | 0      | train
------------------------------------------------------
376 K     Trainable params
0         Non-trainable params
376 K     Total params
1.504     Total estimated model params size (MB)
45        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 8, global step 6993: 'val_loss' was not in top 1


Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_loss did not improve in the last 5 records. Best score: 4.193. Signaling Trainer to stop.
Epoch 9, global step 7770: 'val_loss' was not in top 1


🏃 View run run0 at: http://140.112.106.216:3683/#/experiments/5/runs/1336e0995c404c009b50f9ed4c20c091
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/5


### Inference

In [17]:
# NOTE: the inference model MUST be the same as the training model
best_model_experiment_name = "gcn-exp"
best_model_checkpoint_path = "run0-emb_dim=32-num_layers=3-lr=0.001-batch_size=1024-epochs=50-best-checkpoint-epoch=04-val_loss=4.19.ckpt"
best_model_path = f"test_checkpoints/{best_model_experiment_name}/{best_model_checkpoint_path}"

model = GCNRec.load_from_checkpoint(
    checkpoint_path=best_model_path,
    num_users=TRAIN_NUM_USERS,
    num_items=TRAIN_NUM_ITEMS,
    graph_data=train_graph,
    dim_id=EMB_DIM,
    num_layers=NUM_LAYERS,
    concat=True,
    lr=LR,
)


In [18]:
# start inference
trainer.test(model=model, dataloaders=test_loader)

/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

🏃 View run run0 at: http://140.112.106.216:3683/#/experiments/5/runs/1336e0995c404c009b50f9ed4c20c091
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/5


[{}]

In [19]:
model.test_results

{'user': tensor([   0,    0,    0,  ..., 2063, 2063, 2063]),
 'item': tensor([7977, 4839,  979,  ..., 7982, 5969, 5025]),
 'score': tensor([1843003.6250,  101421.0938, 1435891.6250,  ...,   19326.1875,
           93253.3281,  117403.7344]),
 'label': tensor([1., 0., 1.,  ..., 0., 0., 0.]),
 'user_emb': tensor([[-5.2552e+01, -2.2469e+01, -5.2797e+01,  ..., -3.7981e+01,
          -2.9073e+01, -3.9694e+01],
         [-2.4888e+01, -1.0584e+01, -2.4902e+01,  ..., -1.7977e+01,
          -1.3711e+01, -1.8814e+01],
         [-4.4802e+00, -1.8918e+00, -4.4537e+00,  ..., -3.2318e+00,
          -2.4526e+00, -3.3871e+00],
         ...,
         [-1.4577e+02, -6.2241e+01, -1.4630e+02,  ..., -1.0534e+02,
          -8.0557e+01, -1.1014e+02],
         [-2.6319e+02, -1.1251e+02, -2.6456e+02,  ..., -1.9025e+02,
          -1.4568e+02, -1.9888e+02],
         [-1.9091e-03,  1.1797e-01,  2.9234e-01,  ...,  1.2274e-02,
           9.1983e-02,  6.7818e-02]]),
 'item_emb': tensor([[-1.8439e+03, -1.0309e+03, -3.

In [20]:
eval_df = evaluator.prepare_evaluation_data(model.test_results)
eval_df

,user,rec_items,gt_items
0,0,"[2090, 2419, 281, 4142, 5946, 4932, 316, 3550,...","[7977, 979, 97, 2419, 2090]"
1,1,"[3550, 968, 1364, 7999, 2501, 972, 871, 2444, ...","[3392, 5801, 6570, 8192]"
2,2,"[2419, 1605, 4045, 1344, 945, 991, 8103, 1354,...","[7978, 5768]"
3,3,"[260, 231, 49, 907, 1035, 1009, 7908, 969, 330...","[969, 3246, 3303, 2066, 6880, 7908, 7979, 2932..."
4,4,"[1242, 5941, 8081, 1354, 7599, 4970, 4893, 5, ...","[1492, 1139, 4085, 8160, 6253, 1380, 4395, 8212]"
...,...,...,...
2059,2059,"[870, 1035, 961, 4493, 1326, 4964, 5941, 1354,...","[1267, 2606, 1584, 4493, 1488, 4964, 3500, 836..."
2060,2060,"[688, 1014, 1865, 1344, 5566, 1539, 1275, 621,...","[1596, 1790, 5627, 977, 6914, 8311, 1587, 823,..."
2061,2061,"[49, 688, 5706, 7546, 6170, 942, 1009, 480, 79...","[8140, 8017, 8126, 5941, 8214, 8075]"
2062,2062,"[4932, 3289, 1344, 0, 258, 1326, 7999, 1395, 6...","[3289, 4932, 645, 1496, 1889, 1326]"


In [21]:
eval_score_df = evaluator.evaluate(eval_df, K=5)
eval_score_df = evaluator.evaluate(eval_score_df, K=10)
eval_score_df = evaluator.evaluate(eval_score_df, K=20)
eval_score_df.describe()

,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.337264,0.091020,0.176163,0.397508,0.170115,0.174952,0.437035,0.281546,0.157389
std,595.969798,0.370633,0.155773,0.220108,0.325142,0.213910,0.179938,0.278844,0.263457,0.142845
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,515.750000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.270238,0.076923,0.050000
50%,1031.500000,0.193426,0.003185,0.100000,0.411834,0.100000,0.100000,0.443907,0.214286,0.100000
75%,1547.250000,0.630930,0.125000,0.200000,0.630930,0.250000,0.300000,0.636170,0.428571,0.250000
max,2063.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.900000,1.000000,1.000000,0.800000
